<a href="https://www.kaggle.com/code/wilfriedtsetse04/neurosight-ai-notebook?scriptVersionId=303604334" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/wilfriedtsetse04/neurosight-ai-notebook?scriptVersionId=303586328" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/wilfriedtsetse04/neurosight-ai-notebook?scriptVersionId=300807538" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np

from sklearn.metrics import f1_score, recall_score, roc_auc_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [22]:
# Cell 2 in your notebook
import sys
import os

# Ensure the working directory is in the path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Import directly from the filename shown in your sidebar
from data_pipeline import prepare_data, plot_class_distribution, show_sample_batch

In [25]:
# load dataset with the new pipleine 
CSV_PATH = "/kaggle/input/neurosight-ai-dataset/neurosight-mri-dataset/data.csv"
IMAGES_DIR = "/kaggle/input/neurosight-ai-dataset/neurosight-mri-dataset/OriginalDataset"
train_loader, val_loader, class_names = prepare_data(
    CSV_PATH,
    IMAGES_DIR,
    batch_size=32,
    val_split=0.2
)

print("Classes:", class_names)

TypeError: prepare_data() missing 1 required positional argument: 'images_dir'

In [ ]:
# data set audit 
plot_class_distribution(train_loader.dataset)


# Model Architecture & Training Engine 
## Define the NeuroSight ai model 
#### using transfert learning by loading efficientNetb0 and replace the final classifier head to match our 4 alzheimer stages 

In [ ]:
class NeuroSightModel(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True
        )

        in_features = self.backbone.classifier.in_features

        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
# Initialize model 
model = NeuroSightModel(num_classes=len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()

In [ ]:
# PHASE ! : Warm up training 
for param in model.backbone.parameters():
    param.requires_grad = False

for param in model.backbone.classifier.parameters():
    param.requires_grad = True

In [ ]:
# Optimizer :
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=2,
    factor=0.5
)

# the Training Loop :
#### a loop that handles training and validation and exports the final model .pth 

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f}")

In [ ]:
# Validation function :
def evaluate_model(model, loader):

    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")

    try:
        roc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except:
        roc = 0

    return f1, recall, roc

In [ ]:
# Evaluate After Warm up 
f1, recall, roc = evaluate_model(model, val_loader)

print("Warm-up Results")
print("F1:", f1)
print("Recall:", recall)
print("ROC-AUC:", roc)

In [ ]:
# PHASE 2 : Fine tuning :
for param in model.backbone.parameters():
    param.requires_grad = True
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-2
)

In [ ]:
# Fine tuning training :
epochs_finetune = 10

for epoch in range(epochs_finetune):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    f1, recall, roc = evaluate_model(model, val_loader)

    print(f"Epoch {epoch+1}")
    print("Loss:", running_loss)
    print("F1:", f1)
    print("Recall:", recall)
    print("ROC-AUC:", roc)

In [ ]:
# Save model : 
torch.save(model.state_dict(), "neurosight_model.pth")

print("Model saved")